
# E1 re-run after the D3 interval fix, shard 31 of 32

Re-runs roughly 62 of the 1,983 canonical E1 cells with the
D3 reporting scale corrected.

**What was wrong.** Both engines build an interval as
`theta_hat +/- t * s / sqrt(J)`, but the unpaired design D3 emitted `s` as the
average marginal sd `sqrt((sA^2 + sB^2)/2)` while its own Welch test uses
`SE = sqrt(sA^2 + sB^2) / sqrt(J)`. The interval was therefore narrower than
the design's own test by exactly `sqrt(2)`. In the published grid D3 has a
correct level of 0.0498 at `rho = 0` alongside a coverage of 0.837, which is
the signature of that error rather than a property of unpaired analysis.

**What changes.** Only D3's `ci_cover`. Levels, power and `E[J]` come from
p-values and `J`, which the fix does not touch, so those columns must come back
identical to the previous run; the merge script checks exactly that.

Run all cells, then download `E1_recheck_shard31_of32_results.csv` when prompted. Shards are independent and
can run in parallel sessions; the run cell also parallelises across the cores of
this VM, sized by available RAM.

This shard carries about 580 CPU-seconds of work by the previous
run's timings, so expect at most ~10 min on one core and
~5 min on a typical 2-core Colab VM. Both are upper bounds:
those timings were recorded under a fully loaded local machine, and a measured
re-run of one shard came in at roughly half the predicted cost.


In [ ]:

import multiprocessing as mp
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_URL = "https://github.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA.git"
CLONED_REPO = "/content/TISCA_repo"

if not os.path.isdir(os.path.join(CLONED_REPO, "tisca", "python")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONED_REPO], check=True)
REPO_ROOT = CLONED_REPO
SOURCE_ROOT = os.path.join(REPO_ROOT, "tisca", "python")
assert os.path.isdir(SOURCE_ROOT), f"TISCA package not found at {SOURCE_ROOT}"

# ---- the D3 reporting-scale fix -------------------------------------------------
# Applied to the checkout before the package is imported. Each patch asserts that it
# found either the old text (and rewrites it) or the new text (already upstream); a
# file that matches neither stops the run rather than producing numbers of unknown
# provenance.
PATCHES = [
    (os.path.join(SOURCE_ROOT, "tisca", "outermc", "designs.py"),
     "                 np.sqrt((s2A + s2B) / 2), capped), dict(",
     "                 np.sqrt(s2A + s2B), capped), dict("),
    (os.path.join(SOURCE_ROOT, "tisca", "outermc", "joint_engine.py"),
     "    report_sd = np.sqrt((s2a[:, None] + s2b) / 2.0)",
     "    report_sd = np.sqrt(s2a[:, None] + s2b)"),
]

for path, old, new in PATCHES:
    with open(path) as fh:
        text = fh.read()
    if old in text:
        with open(path, "w") as fh:
            fh.write(text.replace(old, new))
        print("[PATCHED]", os.path.basename(path))
    elif new in text:
        print("[ALREADY FIXED]", os.path.basename(path))
    else:
        raise RuntimeError(
            f"{path}: neither the old nor the fixed reporting scale was found. "
            "The upstream file has changed; regenerate this notebook."
        )

sys.path.insert(0, SOURCE_ROOT)

OUTPUT_ROOT = "/content/TISCA_E1_recheck"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ---- the worker, written to a real module -----------------------------------------
# multiprocessing.Pool sends tasks by pickling the callable by qualified name, so a
# function defined in a notebook cell cannot be a worker even under fork: the child
# fails with "attribute lookup _run_one on __main__ failed". Writing the worker to an
# importable module is what makes the pool usable from a notebook at all.
WORKER_SOURCE = r"""
import os
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.environ["TISCA_SOURCE_ROOT"])
from tisca.outermc import engine, summarize_ocs

_MATRIX = None


# The pre-specified real loss pair driving the two empirical families.
def load_empirical_matrix():
    path = os.path.join(os.environ["TISCA_REPO_ROOT"], "legacy", "Paper_Experiments",
                        "DGP1_500_results.csv")
    raw = pd.read_csv(path)
    matrix = raw[["mvbcf_pehe1", "bcf_pehe1"]].to_numpy(dtype=float)
    matrix = matrix[np.all(np.isfinite(matrix), axis=1)]
    assert matrix.ndim == 2 and matrix.shape[1] == 2 and matrix.shape[0] >= 2
    return matrix


# Execute one grid cell and return its operating-characteristic row.
def run_one(cell):
    global _MATRIX
    cfg = dict(cell["config"])
    if cfg.get("matrix") == "__MATRIX__":
        if _MATRIX is None:
            _MATRIX = load_empirical_matrix()
        cfg["matrix"] = _MATRIX
    t0 = time.time()
    summary, _, _ = engine.run_e1(cfg)
    row = summarize_ocs([summary]).iloc[0].to_dict()
    row.update(cell["factors"])
    row.update(cell_id=cell["cell_id"], module=cell["module"],
               projected_R=cfg["R"], bootstrap_B=cfg.get("B", np.nan),
               cell_seconds=round(time.time() - t0, 3))
    return row
"""

os.environ["TISCA_SOURCE_ROOT"] = SOURCE_ROOT
os.environ["TISCA_REPO_ROOT"] = REPO_ROOT
WORKER_PATH = os.path.join(OUTPUT_ROOT, "e1_worker.py")
with open(WORKER_PATH, "w") as fh:
    fh.write(WORKER_SOURCE)
sys.path.insert(0, OUTPUT_ROOT)

import e1_worker

from tisca.outermc import e1_grid, engine, summarize_ocs
from tisca import multiplicity

print("cpu_count:", os.cpu_count())
print("outputs:", OUTPUT_ROOT)



## Self-check: reproduce the defect and its repair

Before any grid compute, confirm numerically that the two interval scales differ
the way the diagnosis says they do. This is a closed-form prediction, so a
failure here means the diagnosis is wrong, not that the run was unlucky.


In [ ]:

# Reproduce the defect and its repair before spending any compute. With equal group
# sizes the two interval scales differ by exactly sqrt(2), so at rho = 0 -- where an
# unpaired analysis of paired rows is valid and the Welch test is correctly sized --
# the as-coded interval must land near 2*Phi(1.96/sqrt(2)) - 1 = 0.834 while the
# consistent interval must land near 0.95. If that separation does not appear, the
# patch above did not do what this notebook claims it does.
from scipy import stats

rng = np.random.default_rng(0)
R_CHK, J_CHK = 200_000, 60
crit = stats.t.ppf(0.975, J_CHK - 1)
print(f"{'rho':>5} {'as-coded':>10} {'consistent':>12}")
checked = {}
for rho in (-0.3, 0.0, 0.6):
    cov = np.array([[1.0, rho], [rho, 1.0]])
    x = rng.multivariate_normal([0.0, 0.0], cov, size=(R_CHK, J_CHK))
    A, B = x[..., 0], x[..., 1]
    s2A, s2B = A.var(1, ddof=1), B.var(1, ddof=1)
    dbar = A.mean(1) - B.mean(1)
    as_coded = float(np.mean(np.abs(dbar) <= crit * np.sqrt((s2A + s2B) / 2) / np.sqrt(J_CHK)))
    consistent = float(np.mean(np.abs(dbar) <= crit * np.sqrt((s2A + s2B) / J_CHK)))
    checked[rho] = (as_coded, consistent)
    print(f"{rho:>5.1f} {as_coded:>10.4f} {consistent:>12.4f}")

assert abs(checked[0.0][0] - 0.834) < 0.01, checked[0.0]
assert abs(checked[0.0][1] - 0.950) < 0.01, checked[0.0]
assert checked[0.6][1] > 0.99, checked[0.6]     # over-covers at positive rho
assert checked[-0.3][1] < 0.93, checked[-0.3]   # under-covers at negative rho
print("[PASS] the sqrt(2) artefact and its repair both reproduce")



## The canonical grid and this shard's slice

Cells are imported from `tisca/python/tisca/outermc/e1_grid.py`, the single
source of truth for all 1,983 cells. Only the partition is listed below, and it
is balanced by measured running time rather than by cell count: Module A averages
~13 s per cell against Module B's ~1 s, so equal-sized shards would not be
equal-length shards.


In [ ]:
SHARD_INDEX = 30
N_SHARDS = 32
OUTPUT_NAME = "E1_recheck_shard31_of32_results.csv"
EXPECTED_CPU_SECONDS = 579.9
SHARD_CELL_IDS = [
    'A_0481',
    'A_0307',
    'A_0553',
    'A_0519',
    'A_0558',
    'A_0575',
    'A_0734',
    'D_0086',
    'B_0617',
    'D_0101',
    'D_0213',
    'D_0198',
    'C_0099',
    'A_0626',
    'A_0305',
    'C_0134',
    'C_0079',
    'A_0641',
    'A_0211',
    'D_0052',
    'D_0038',
    'C_0174',
    'D_0039',
    'D_0176',
    'C_0152',
    'C_0013',
    'A_0434',
    'A_0835',
    'A_0777',
    'A_0460',
    'A_0181',
    'B_0079',
    'C_0214',
    'A_0859',
    'A_0838',
    'A_0412',
    'A_0371',
    'D_0123',
    'D_0009',
    'C_0083',
    'B_0190',
    'A_0084',
    'B_0263',
    'B_0223',
    'B_0509',
    'B_0479',
    'B_0102',
    'B_0564',
    'B_0502',
    'B_0159',
    'D_0035',
    'A_0065',
    'B_0315',
    'B_0489',
    'B_0513',
    'B_0024',
    'B_0506',
    'B_0417',
    'B_0539',
    'B_0647',
    'B_0176',
    'B_0656',
]
assert len(SHARD_CELL_IDS) == len(set(SHARD_CELL_IDS))


In [ ]:

MATRIX = e1_worker.load_empirical_matrix()

# The grid is imported, never retyped: e1_grid.py is the single source of truth for
# all 1,983 cells and the local runner reads the same object.
GRID_ALL = e1_grid.make_grid("ABCD", matrix="__MATRIX__",
                             planning_alpha=multiplicity.planning_alpha)
for _m, _n in e1_grid.EXPECTED.items():
    _got = sum(c["module"] == _m for c in GRID_ALL)
    assert _got == _n, f"module {_m}: {_got} cells, expected {_n}"


def resolve_oracle_sigma(grid, matrix):
    """Fill each cell's true sigma_D once, in the parent.

    engine.sigma_D_true estimates the non-normal families' sigma from a 1,000,000-draw
    sample and caches inside one process, so leaving the call to the workers repeats
    every estimate in every worker.
    """
    table = {}
    for cell in grid:
        cfg = cell["config"]
        key = (cfg["family"], cfg["rho"], cfg["sigma_a"], cfg["sigma_b"])
        if key not in table:
            probe = dict(cfg)
            probe["matrix"] = matrix if cfg["matrix"] == "__MATRIX__" else cfg["matrix"]
            table[key] = engine.sigma_D_true(probe)
        cfg["sigma_D"] = table[key]
    return table


_table = resolve_oracle_sigma(GRID_ALL, MATRIX)
print(f"resolved {len(_table)} oracle sigma_D values")

# SHARD_CELL_IDS is the explicit, time-balanced slice this session owns (see
# _assign_shards in the generator). The grid itself is still imported rather than
# retyped; only the partition is listed here, so a missing or duplicated cell is
# caught by the assertion below and again by the merge script across all shards.
_by_id = {c["cell_id"]: c for c in GRID_ALL}
missing = sorted(set(SHARD_CELL_IDS) - set(_by_id))
assert not missing, f"shard names {len(missing)} cells absent from the grid: {missing[:3]}"
SHARD = [_by_id[cid] for cid in SHARD_CELL_IDS]
print(f"shard {SHARD_INDEX + 1}/{N_SHARDS}: {len(SHARD)} of {len(GRID_ALL)} cells")
print("by module:", {m: sum(c["module"] == m for c in SHARD) for m in "ABCD"})
print(f"budgeted work: {EXPECTED_CPU_SECONDS:.0f} CPU-s "
      f"(<= ~{EXPECTED_CPU_SECONDS / 60:.0f} min at 1 core; an upper bound, the "
      f"source timings were recorded under load)")



## Run the shard

One checkpointed row per completed cell, so an interrupted session loses at most
the cells in flight; re-running this cell resumes.


In [ ]:

def available_gb():
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 / 1024
    except OSError:
        pass
    return float("inf")


# Each worker holds an (R, Jmax, 2) block plus working copies: measured peak RSS is
# ~0.52 GB on the heaviest cells, so the pool is capped by memory as well as cores.
# A 40-core VM that budgets by cores alone will swap.
MEM_PER_WORKER = 0.6
by_mem = max(1, int((available_gb() * 0.7) // MEM_PER_WORKER))
JOBS = max(1, min(os.cpu_count() or 2, by_mem))
print(f"{available_gb():.1f} GB available -> {JOBS} workers "
      f"(cpu {os.cpu_count()}, memory cap {by_mem})")

OUTPUT_FILE = os.path.join(OUTPUT_ROOT, OUTPUT_NAME)

rows = []
done = set()
if os.path.exists(OUTPUT_FILE) and os.path.getsize(OUTPUT_FILE) > 0:
    previous = pd.read_csv(OUTPUT_FILE)
    rows = previous.to_dict("records")
    done = set(previous["cell_id"].astype(str))
    print(f"resuming: {len(done)} cells already checkpointed")
pending = [c for c in SHARD if c["cell_id"] not in done]
print(f"{len(pending)} cells pending")

started = time.time()
# The checkpoint rewrites the whole frame rather than appending one row at a time.
# Module-B rows carry factor columns (K, correction) that Module-A rows do not, so
# appending each row in its own column order silently misaligns the CSV: values land
# under the wrong headers and cell_id stops matching. Building one DataFrame unifies
# the columns, and a few hundred rows are cheap enough to rewrite per cell.
if pending:
    with mp.get_context("fork").Pool(JOBS) as pool:
        for row in tqdm(pool.imap_unordered(e1_worker.run_one, pending, chunksize=1),
                        total=len(pending), unit="cell"):
            rows.append(row)
            pd.DataFrame(rows).to_csv(OUTPUT_FILE, index=False)
print(f"elapsed {time.time() - started:.0f}s")


## Completeness check and download

In [ ]:

RESULTS = pd.read_csv(OUTPUT_FILE)
expected = {c["cell_id"] for c in SHARD}
got = set(RESULTS["cell_id"].astype(str))
missing = sorted(expected - got)
assert not missing, f"{len(missing)} cells missing, first={missing[:3]}"
assert not RESULTS["cell_id"].duplicated().any(), "duplicate cell_id in checkpoint"
print(f"[PASS] {len(RESULTS)} rows, all {len(expected)} shard cells present")

# The fix moves D3 coverage and nothing else, so D3 null cells are the one place the
# change should be visible in this shard.
d3_null = RESULTS[(RESULTS["design"] == "D3") & (RESULTS["theta"] == 0)]
if len(d3_null):
    print("\nD3 null cells, coverage by rho (was 0.79-0.99 rising in rho, "
          "artefactually centred near 0.90):")
    print(d3_null.groupby("rho")[["ci_cover", "reject_rate"]].mean().round(4))

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("Downloaded:", OUTPUT_FILE)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
